# Phase 4 — Data Integration & Modelling

## 4.1 Initialise Integration Environment

The validated cleaned datasets are loaded from the processed output layer for
data integration and modelling.

This phase combines related business entities using validated keys while
controlling join cardinality, row counts, and potential duplication.

In [1]:
from pathlib import Path
import pandas as pd

# Define the cleaned data location
output_data_path = Path("../data/output")

# Identify all cleaned CSV files
cleaned_files = list(output_data_path.glob("*_cleaned.csv"))

# Load cleaned datasets
datasets = {}

for file in cleaned_files:
    dataset_name = file.stem.replace("_cleaned", "")
    datasets[dataset_name] = pd.read_csv(file)

print(f"Successfully loaded {len(datasets)} cleaned datasets.")

Successfully loaded 11 cleaned datasets.


## 4.2 Confirm Integration Inputs

The cleaned input datasets are reviewed to confirm that the expected processed
tables are available before integration begins.

In [2]:
# Confirm cleaned datasets available for integration

integration_inputs = []

for name, df in datasets.items():
    integration_inputs.append({
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns)
    })

integration_inputs_df = pd.DataFrame(integration_inputs)

display(
    integration_inputs_df.sort_values("dataset")
)

,dataset,rows,columns
0,closed_deals_dataset,842,14
1,customers_dataset,99441,5
2,geolocation_dataset,738332,5
3,marketing_qualified_leads_dataset,8000,4
5,order_items_dataset,112650,7
6,order_payments_dataset,103886,6
7,order_reviews_dataset,99224,7
4,orders_dataset,99441,9
9,product_category_name_translation,71,2
8,products_dataset,32951,10


## 4.3 Define Analytical Grain and Integration Model

The primary analytical dataset will use **order-item grain**, where each row
represents one product item within an order.

The `order_items_dataset` is therefore used as the central transactional
dataset. This grain preserves product, seller, price, and freight-level detail
required for downstream analysis.

Related datasets will be integrated according to their relationship with the
order-item grain.

### Integration Model

- `order_items_dataset` → base transactional dataset
- `orders_dataset` → order and customer information
- `customers_dataset` → customer attributes
- `products_dataset` → product attributes
- `product_category_name_translation` → English product category
- `sellers_dataset` → seller attributes

Datasets containing potentially multiple records per order, such as payments
and reviews, will be assessed and aggregated to order level before joining to
the order-item dataset where necessary. This prevents unintended many-to-many
row multiplication.

## 4.4 Validate Order-Item Grain

The order-items dataset is validated as the base transactional table.

The combination of `order_id` and `order_item_id` is expected to uniquely
identify each order-item record.

In [3]:
# Validate the proposed order-item grain

order_items = datasets["order_items_dataset"]

grain_validation = pd.DataFrame({
    "metric": [
        "Total rows",
        "Unique orders",
        "Unique order-item combinations",
        "Duplicate order-item combinations"
    ],
    "value": [
        len(order_items),
        order_items["order_id"].nunique(),
        order_items[["order_id", "order_item_id"]]
            .drop_duplicates()
            .shape[0],
        order_items.duplicated(
            subset=["order_id", "order_item_id"]
        ).sum()
    ]
})

display(grain_validation)

,metric,value
0,Total rows,112650
1,Unique orders,98666
2,Unique order-item combinations,112650
3,Duplicate order-item combinations,0


## 4.5 Validate Relationship Cardinality

The cardinality of the proposed integration relationships is validated before
datasets are joined.

For each relationship, the uniqueness of the parent key is assessed to confirm
whether the join can safely be performed as a many-to-one relationship without
unexpected row multiplication.

In [4]:
# Validate cardinality of planned relationships

relationship_checks = [
    ("order_items → orders",
     "orders_dataset", "order_id"),

    ("orders → customers",
     "customers_dataset", "customer_id"),

    ("order_items → products",
     "products_dataset", "product_id"),

    ("order_items → sellers",
     "sellers_dataset", "seller_id"),

    ("products → category translation",
     "product_category_name_translation", "product_category_name")
]

cardinality_results = []

for relationship, dataset_name, key in relationship_checks:

    df = datasets[dataset_name]

    total_rows = len(df)
    unique_keys = df[key].nunique()
    duplicate_keys = df.duplicated(subset=[key]).sum()

    cardinality_results.append({
        "relationship": relationship,
        "parent_dataset": dataset_name,
        "parent_key": key,
        "rows": total_rows,
        "unique_keys": unique_keys,
        "duplicate_keys": duplicate_keys,
        "safe_many_to_one": duplicate_keys == 0
    })

cardinality_df = pd.DataFrame(cardinality_results)

display(cardinality_df)

,relationship,parent_dataset,parent_key,rows,unique_keys,duplicate_keys,safe_many_to_one
0,order_items → orders,orders_dataset,order_id,99441,99441,0,True
1,orders → customers,customers_dataset,customer_id,99441,99441,0,True
2,order_items → products,products_dataset,product_id,32951,32951,0,True
3,order_items → sellers,sellers_dataset,seller_id,3095,3095,0,True
4,products → category translation,product_category_name_translation,product_category_name,71,71,0,True


## 4.6 Integrate Order Information

The order-items dataset is joined to the orders dataset using `order_id`.

A left join is used to preserve the complete order-item grain. The relationship
is explicitly validated as many-to-one to prevent unexpected row multiplication.

Post-join validation confirms that the number of order-item records remains
unchanged and identifies any unmatched order records.

In [5]:
# Join order items to orders

order_items_df = datasets["order_items_dataset"]
orders_df = datasets["orders_dataset"]

rows_before = len(order_items_df)

integrated_df = order_items_df.merge(
    orders_df,
    on="order_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

rows_after = len(integrated_df)

print(f"Rows before join : {rows_before:,}")
print(f"Rows after join  : {rows_after:,}")
print(f"Row difference   : {rows_after - rows_before:,}")

print("\nJoin result:")
print(integrated_df["_merge"].value_counts())

Rows before join : 112,650
Rows after join  : 112,650
Row difference   : 0

Join result:
_merge
both          112650
left_only          0
right_only         0
Name: count, dtype: int64


## 4.7 Integrate Customer Information

Customer attributes are added to the integrated order-item dataset using
`customer_id`.

The customer dataset contains one record per `customer_id`, allowing the
relationship to be validated as many-to-one. A left join preserves the
established order-item analytical grain.

Post-join checks confirm that all customer references were matched and that
the integration did not introduce additional rows.

In [6]:
# Remove the previous merge indicator
integrated_df = integrated_df.drop(columns="_merge")

# Join customer information

customers_df = datasets["customers_dataset"]

rows_before = len(integrated_df)

integrated_df = integrated_df.merge(
    customers_df,
    on="customer_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

rows_after = len(integrated_df)

print(f"Rows before join : {rows_before:,}")
print(f"Rows after join  : {rows_after:,}")
print(f"Row difference   : {rows_after - rows_before:,}")

print("\nJoin result:")
print(integrated_df["_merge"].value_counts())

Rows before join : 112,650
Rows after join  : 112,650
Row difference   : 0

Join result:
_merge
both          112650
left_only          0
right_only         0
Name: count, dtype: int64


## 4.8 Integrate Product Information

Product attributes are added to the integrated order-item dataset using
`product_id`.

The products dataset contains one record per `product_id`, allowing the
relationship to be validated as many-to-one. A left join preserves the
established order-item analytical grain.

Post-join validation confirms whether all product references were successfully
matched and ensures that no unintended row multiplication occurred.

In [7]:
# Remove the previous merge indicator
integrated_df = integrated_df.drop(columns="_merge")

# Join product information

products_df = datasets["products_dataset"]

rows_before = len(integrated_df)

integrated_df = integrated_df.merge(
    products_df,
    on="product_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

rows_after = len(integrated_df)

print(f"Rows before join : {rows_before:,}")
print(f"Rows after join  : {rows_after:,}")
print(f"Row difference   : {rows_after - rows_before:,}")

print("\nJoin result:")
print(integrated_df["_merge"].value_counts())

Rows before join : 112,650
Rows after join  : 112,650
Row difference   : 0

Join result:
_merge
both          112650
left_only          0
right_only         0
Name: count, dtype: int64


## 4.9 Integrate Product Category Translation

English product category names are added using the product category translation
dataset.

The relationship uses `product_category_name` as the join key. A left join is
used to preserve the established order-item grain while allowing products with
missing or untranslated category values to remain in the analytical dataset.

Post-join validation confirms that the integration does not introduce
additional rows and identifies records without an English category translation.

In [8]:
# Remove the previous merge indicator
integrated_df = integrated_df.drop(columns="_merge")

# Join product category translation

category_df = datasets["product_category_name_translation"]

rows_before = len(integrated_df)

integrated_df = integrated_df.merge(
    category_df,
    on="product_category_name",
    how="left",
    validate="many_to_one",
    indicator=True
)

rows_after = len(integrated_df)

print(f"Rows before join : {rows_before:,}")
print(f"Rows after join  : {rows_after:,}")
print(f"Row difference   : {rows_after - rows_before:,}")

print("\nJoin result:")
print(integrated_df["_merge"].value_counts())

Rows before join : 112,650
Rows after join  : 112,650
Row difference   : 0

Join result:
_merge
both          111023
left_only       1627
right_only         0
Name: count, dtype: int64


## 4.10 Investigate Unmatched Product Categories

Order-item records without a matching English product category translation are
investigated to determine whether the source product category is missing or
whether a category exists but is absent from the translation reference dataset.

This distinction determines whether the unmatched records represent missing
source information or incomplete reference-data coverage.

In [9]:
# Investigate unmatched product category translations

unmatched_categories = integrated_df[
    integrated_df["_merge"] == "left_only"
].copy()

missing_source_category = (
    unmatched_categories["product_category_name"].isna().sum()
)

category_present_no_translation = (
    unmatched_categories["product_category_name"].notna().sum()
)

print(f"Total unmatched records              : {len(unmatched_categories):,}")
print(f"Missing source product category      : {missing_source_category:,}")
print(f"Category present but no translation  : {category_present_no_translation:,}")

# Show any categories that exist but have no translation
untranslated_categories = (
    unmatched_categories.loc[
        unmatched_categories["product_category_name"].notna(),
        "product_category_name"
    ]
    .value_counts()
    .reset_index()
)

untranslated_categories.columns = [
    "product_category_name",
    "order_item_count"
]

display(untranslated_categories)

Total unmatched records              : 1,627
Missing source product category      : 1,603
Category present but no translation  : 24


,product_category_name,order_item_count
0,portateis_cozinha_e_preparadores_de_alimentos,15
1,pc_gamer,9


## 4.11 Handle Unmatched Product Categories

Unmatched product-category translations are retained rather than removed.

Records with no source product category are distinguished from records where a
source category exists but no corresponding English translation is available.
Quality flags are created to preserve this distinction for downstream analysis.

No category values are artificially imputed because a reliable source value is
not available for all affected records.

In [10]:
# Create product category quality flags

integrated_df["dq_missing_product_category"] = (
    integrated_df["product_category_name"].isna()
)

integrated_df["dq_missing_category_translation"] = (
    integrated_df["product_category_name"].notna()
    & integrated_df["product_category_name_english"].isna()
)

category_quality_summary = pd.DataFrame({
    "quality_check": [
        "Missing source product category",
        "Missing English category translation"
    ],
    "records": [
        integrated_df["dq_missing_product_category"].sum(),
        integrated_df["dq_missing_category_translation"].sum()
    ]
})

display(category_quality_summary)

,quality_check,records
0,Missing source product category,1603
1,Missing English category translation,24


## 4.12 Integrate Seller Information

Seller attributes are added to the analytical dataset using `seller_id`.

The sellers dataset contains one record per seller, allowing the relationship
to be validated as many-to-one. A left join preserves the established
order-item grain and enables unmatched seller references to be identified.

In [11]:
# Join seller information

sellers_df = datasets["sellers_dataset"]

rows_before = len(integrated_df)

integrated_df = integrated_df.merge(
    sellers_df,
    on="seller_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

rows_after = len(integrated_df)

print(f"Rows before join : {rows_before:,}")
print(f"Rows after join  : {rows_after:,}")
print(f"Row difference   : {rows_after - rows_before:,}")

print("\nJoin result:")
print(integrated_df["_merge"].value_counts())

ValueError: Cannot use name of an existing column for indicator column

In [12]:
# Remove the previous merge indicator
integrated_df = integrated_df.drop(columns="_merge")

# Join seller information

sellers_df = datasets["sellers_dataset"]

rows_before = len(integrated_df)

integrated_df = integrated_df.merge(
    sellers_df,
    on="seller_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

rows_after = len(integrated_df)

print(f"Rows before join : {rows_before:,}")
print(f"Rows after join  : {rows_after:,}")
print(f"Row difference   : {rows_after - rows_before:,}")

print("\nJoin result:")
print(integrated_df["_merge"].value_counts())

Rows before join : 112,650
Rows after join  : 112,650
Row difference   : 0

Join result:
_merge
both          112650
left_only          0
right_only         0
Name: count, dtype: int64


## 4.13 Analyse Payment Cardinality

The payment dataset is assessed before integration because an order may contain
multiple payment records.

Directly joining multiple payment records to an order-item-level dataset could
create a many-to-many relationship and unintentionally multiply transactional
rows.

Payment records are therefore analysed at order level before an appropriate
aggregation strategy is applied.

In [13]:
# Analyse payment records per order

payments_df = datasets["order_payments_dataset"]

payment_cardinality = (
    payments_df
    .groupby("order_id")
    .size()
    .reset_index(name="payment_record_count")
)

print(f"Payment rows                  : {len(payments_df):,}")
print(f"Unique orders with payments   : {payments_df['order_id'].nunique():,}")
print(f"Maximum payments for one order: {payment_cardinality['payment_record_count'].max():,}")

print(
    f"Orders with multiple payments : "
    f"{(payment_cardinality['payment_record_count'] > 1).sum():,}"
)

display(
    payment_cardinality
    .sort_values("payment_record_count", ascending=False)
    .head(10)
)

Payment rows                  : 103,886
Unique orders with payments   : 99,440
Maximum payments for one order: 29
Orders with multiple payments : 2,961


,order_id,payment_record_count
97261,fa65dad1b0e818e3ccc5cb0e39231352,29
79611,ccf804e764ed5650cd8759557269dc13,26
15577,285c2e15bebd4ac83635ccc563dc71f4,22
53168,895ab968e7bb0d5659d16cd74cd1650c,21
99020,fedcd9f7ccdc8cba3a18defedd1a5547,19
92724,ee9ca989fc93ba09a6eddc250ce01742,19
29586,4bfcba9e084f46c8e3cb49b0fa6e6159,15
12924,21577126c19bf11a0b91592e5844ba78,15
27411,4689b1816de42507a7d63a4617383c59,14
23431,3c58bffb70dcf45f12bdf66a3c215905,14


## 4.14 Analyse Payment Methods per Order

Orders containing multiple payment records are analysed to determine whether
they also contain multiple payment methods.

This assessment informs the aggregation strategy required to transform the
payment dataset to one record per order before integration with the order-item
analytical model.

In [14]:
# Analyse payment methods used per order

payment_type_profile = (
    payments_df
    .groupby("order_id")
    .agg(
        payment_record_count=("payment_sequential", "count"),
        payment_type_count=("payment_type", "nunique"),
        total_payment_value=("payment_value", "sum"),
        max_installments=("payment_installments", "max")
    )
    .reset_index()
)

print(
    f"Orders with multiple payment records : "
    f"{(payment_type_profile['payment_record_count'] > 1).sum():,}"
)

print(
    f"Orders using multiple payment types  : "
    f"{(payment_type_profile['payment_type_count'] > 1).sum():,}"
)

display(
    payment_type_profile
    .sort_values(
        ["payment_type_count", "payment_record_count"],
        ascending=False
    )
    .head(10)
)

Orders with multiple payment records : 2,961
Orders using multiple payment types  : 2,246


,order_id,payment_record_count,payment_type_count,total_payment_value,max_installments
79611,ccf804e764ed5650cd8759557269dc13,26,2,62.68,1
15577,285c2e15bebd4ac83635ccc563dc71f4,22,2,40.85,1
92724,ee9ca989fc93ba09a6eddc250ce01742,19,2,82.73,1
99020,fedcd9f7ccdc8cba3a18defedd1a5547,19,2,205.74,1
29586,4bfcba9e084f46c8e3cb49b0fa6e6159,15,2,740.76,1
27411,4689b1816de42507a7d63a4617383c59,14,2,529.55,1
45004,73df5d6adbeea12c8ae03df93f346e86,13,2,186.92,1
10261,1a611328643ae11146ba09a4425d2e12,12,2,69.97,1
11497,1d9a9731b9c10fc9cba74e6f74782e8b,12,2,63.58,1
27340,465c2e1bee4561cb39e0db8c5993aafc,12,2,879.44,5


## 4.15 Aggregate Payments to Order Level

The payment dataset contains multiple payment records for some orders, including
orders that use more than one payment method.

To prevent row multiplication during integration, payment records are aggregated
to one record per order.

The aggregation retains the total payment value, number of payment records,
number of payment methods, maximum instalment count, and the payment methods
used for each order.

In [15]:
# Aggregate payment records to one row per order

payment_summary = (
    payments_df
    .groupby("order_id")
    .agg(
        total_payment_value=("payment_value", "sum"),
        payment_record_count=("payment_sequential", "count"),
        payment_type_count=("payment_type", "nunique"),
        max_payment_installments=("payment_installments", "max"),
        payment_types=(
            "payment_type",
            lambda x: " | ".join(sorted(x.dropna().unique()))
        )
    )
    .reset_index()
)

print(f"Original payment rows    : {len(payments_df):,}")
print(f"Aggregated payment rows  : {len(payment_summary):,}")
print(f"Unique payment order IDs : {payment_summary['order_id'].nunique():,}")
print(
    f"Duplicate order IDs      : "
    f"{payment_summary['order_id'].duplicated().sum():,}"
)

display(payment_summary.head(10))

Original payment rows    : 103,886
Aggregated payment rows  : 99,440
Unique payment order IDs : 99,440
Duplicate order IDs      : 0


,order_id,total_payment_value,payment_record_count,payment_type_count,max_payment_installments,payment_types
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1,1,2,credit_card
1,00018f77f2f0320c557190d7a144bdd3,259.83,1,1,3,credit_card
2,000229ec398224ef6ca0657da4fc703e,216.87,1,1,5,credit_card
3,00024acbcdf0a6daa1e931b038114c75,25.78,1,1,2,credit_card
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1,1,3,credit_card
5,00048cc3ae777c65dbb7d2a0634bc1ea,34.59,1,1,1,boleto
6,00054e8431b9d7675808bcb819fb4a32,31.75,1,1,1,credit_card
7,000576fe39319847cbb9d288c5617fa6,880.75,1,1,10,credit_card
8,0005a1a1728c9d785b8e2b08b904576c,157.60,1,1,3,credit_card
9,0005f50442cb953dcd1d21e1fb923495,65.39,1,1,1,credit_card


## 4.16 Integrate Aggregated Payment Data

The order-level payment summary is joined to the integrated order-item dataset.

Because the payment data has been aggregated to one record per order, the join
preserves the order-item grain and avoids row multiplication caused by orders
containing multiple payment transactions.

A left join is used so that order items are retained even where no corresponding
payment record exists.

In [16]:
# Remove previous merge indicator
if "_merge" in integrated_df.columns:
    integrated_df = integrated_df.drop(columns="_merge")

# Join aggregated payment information

rows_before = len(integrated_df)

integrated_df = integrated_df.merge(
    payment_summary,
    on="order_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

rows_after = len(integrated_df)

print(f"Rows before join : {rows_before:,}")
print(f"Rows after join  : {rows_after:,}")
print(f"Row difference   : {rows_after - rows_before:,}")

print("\nJoin result:")
print(integrated_df["_merge"].value_counts())

Rows before join : 112,650
Rows after join  : 112,650
Row difference   : 0

Join result:
_merge
both          112647
left_only          3
right_only         0
Name: count, dtype: int64


## 4.17 Investigate Orders Without Payment Records

The payment integration identified a small number of order-item records without
matching payment information.

These records are examined to determine their order status and other relevant
attributes before deciding how they should be handled in the analytical model.

The records are retained during investigation rather than being removed
automatically.

In [17]:
# Investigate order-item records with no matching payment information

unmatched_payments = integrated_df[
    integrated_df["_merge"] == "left_only"
].copy()

print(f"Unmatched order-item rows : {len(unmatched_payments):,}")
print(f"Unique unmatched orders   : {unmatched_payments['order_id'].nunique():,}")

columns_to_show = [
    "order_id",
    "order_item_id",
    "customer_id",
    "order_status",
    "product_id",
    "price",
    "freight_value"
]

display(
    unmatched_payments[columns_to_show]
)

Unmatched order-item rows : 3
Unique unmatched orders   : 1


,order_id,order_item_id,customer_id,order_status,product_id,price,freight_value
84389,bfbd0f9bdef84302105ad712db648a6c,1,86dc2ffce2dfff336de2f386a786e574,delivered,5a6b04657a4c5ee34285d1e4619a96b4,44.99,2.83
84390,bfbd0f9bdef84302105ad712db648a6c,2,86dc2ffce2dfff336de2f386a786e574,delivered,5a6b04657a4c5ee34285d1e4619a96b4,44.99,2.83
84391,bfbd0f9bdef84302105ad712db648a6c,3,86dc2ffce2dfff336de2f386a786e574,delivered,5a6b04657a4c5ee34285d1e4619a96b4,44.99,2.83


### Finding

The three unmatched order-item records relate to a single order containing
three items.

The order is marked as `delivered`, but no corresponding payment record exists
in the payment dataset. This is therefore treated as a source-data quality
exception rather than expected transactional behaviour.

The order is retained in the analytical dataset to preserve the source
transaction history. Its payment attributes remain null, allowing the exception
to be identified during downstream analysis rather than silently removing the
record.

### 4.18 — Create Missing Payment Quality Flag

In [23]:
# Create data quality flag for orders without payment information

integrated_df["dq_missing_payment"] = (
    ~integrated_df["order_id"].isin(payment_summary["order_id"])
)

print(
    f"Order-item rows flagged with missing payment : "
    f"{integrated_df['dq_missing_payment'].sum():,}"
)

print(
    f"Unique orders affected                      : "
    f"{integrated_df.loc[integrated_df['dq_missing_payment'], 'order_id'].nunique():,}"
)

Order-item rows flagged with missing payment : 3
Unique orders affected                      : 1


## 4.19 Analyse Order Review Cardinality

The order review dataset is assessed before integration to determine whether
multiple review records can exist for the same order.

This validation is required because directly joining multiple reviews to an
order-item-level dataset could multiply rows and change the intended analytical
grain.

In [24]:
# Analyse review records per order

reviews_df = datasets["order_reviews_dataset"]

review_cardinality = (
    reviews_df
    .groupby("order_id")
    .size()
    .reset_index(name="review_record_count")
)

print(f"Review rows                    : {len(reviews_df):,}")
print(f"Unique orders with reviews     : {reviews_df['order_id'].nunique():,}")
print(f"Unique review IDs              : {reviews_df['review_id'].nunique():,}")
print(f"Maximum reviews for one order  : {review_cardinality['review_record_count'].max():,}")
print(
    f"Orders with multiple reviews   : "
    f"{(review_cardinality['review_record_count'] > 1).sum():,}"
)

display(
    review_cardinality
    .sort_values("review_record_count", ascending=False)
    .head(10)
)

Review rows                    : 99,224
Unique orders with reviews     : 98,673
Unique review IDs              : 98,410
Maximum reviews for one order  : 3
Orders with multiple reviews   : 547


,order_id,review_record_count
1455,03c939fd7fd3b38f8485a0f95798f1f6,3
54489,8e17072ec97ce29f0e1f111e598b0c85,3
77319,c88b1d1b157a9999ce368f218a407141,3
86232,df56136b8031ecd28e200bb18e6ddb2e,3
48918,7f13a20e25350f4a55fb2a7c9a2e8d88,2
26449,44aa72f4eebc8c0ff7a3636f6369201b,2
47596,7b994fc7c7528d2098eebcb19d18ce7e,2
20315,34de5f7952fbd04a9a1e5d7001b014b5,2
38156,6318d8c5f6857de384c0b3186d036457,2
3765,09a38776c4e31230864f867821174daa,2


## 4.20 Investigate Orders with Multiple Reviews

Orders containing more than one review record are examined before aggregation.

The investigation determines whether multiple records represent duplicate review
identifiers or genuinely distinct reviews associated with the same order. This
supports an appropriate transformation strategy while preserving meaningful
customer feedback.

In [25]:
# Identify orders containing multiple review records

multi_review_orders = review_cardinality.loc[
    review_cardinality["review_record_count"] > 1,
    "order_id"
]

multiple_reviews = reviews_df[
    reviews_df["order_id"].isin(multi_review_orders)
].copy()

review_investigation = (
    multiple_reviews
    .groupby("order_id")
    .agg(
        review_records=("review_id", "size"),
        unique_review_ids=("review_id", "nunique"),
        unique_scores=("review_score", "nunique"),
        min_score=("review_score", "min"),
        max_score=("review_score", "max")
    )
    .reset_index()
)

print(f"Orders investigated                : {len(review_investigation):,}")
print(
    f"Orders with multiple review IDs    : "
    f"{(review_investigation['unique_review_ids'] > 1).sum():,}"
)
print(
    f"Orders with different review scores: "
    f"{(review_investigation['unique_scores'] > 1).sum():,}"
)

display(
    review_investigation
    .sort_values(
        ["review_records", "unique_review_ids"],
        ascending=False
    )
    .head(20)
)

Orders investigated                : 547
Orders with multiple review IDs    : 547
Orders with different review scores: 202


,order_id,review_records,unique_review_ids,unique_scores,min_score,max_score
8,03c939fd7fd3b38f8485a0f95798f1f6,3,3,2,3,4
323,8e17072ec97ce29f0e1f111e598b0c85,3,3,1,1,1
443,c88b1d1b157a9999ce368f218a407141,3,3,2,3,5
486,df56136b8031ecd28e200bb18e6ddb2e,3,3,1,5,5
0,0035246a40f520710769010f752e7507,2,2,1,5,5
1,013056cfe49763c6f66bda03396c5ee3,2,2,2,4,5
2,0176a6846bcb3b0d3aa3116a9a768597,2,2,1,5,5
3,02355020fd0a40a0d56df9f6ff060413,2,2,2,1,3
4,029863af4b968de1e5d6a82782e662f5,2,2,2,4,5
5,02e0b68852217f5715fb9cc885829454,2,2,1,4,4


## 4.21 Aggregate Reviews to Order Level

The review dataset contains genuinely distinct multiple reviews for some orders.
All 547 orders with multiple review records contain multiple review identifiers,
and 202 of these orders contain differing review scores.

To preserve this information while maintaining the order-item grain of the
analytical dataset, reviews are aggregated to one record per order.

The aggregation retains the number of reviews, average review score, minimum
score, maximum score, and indicators showing whether review comments are
available.

In [26]:
# Aggregate reviews to one row per order

review_summary = (
    reviews_df
    .groupby("order_id")
    .agg(
        review_count=("review_id", "size"),
        unique_review_count=("review_id", "nunique"),
        average_review_score=("review_score", "mean"),
        min_review_score=("review_score", "min"),
        max_review_score=("review_score", "max"),
        reviews_with_comments=(
            "review_comment_message",
            lambda x: x.notna().sum()
        )
    )
    .reset_index()
)

# Round average score for reporting
review_summary["average_review_score"] = (
    review_summary["average_review_score"].round(2)
)

print(f"Original review rows     : {len(reviews_df):,}")
print(f"Aggregated review rows   : {len(review_summary):,}")
print(f"Unique review order IDs  : {review_summary['order_id'].nunique():,}")
print(
    f"Duplicate order IDs      : "
    f"{review_summary['order_id'].duplicated().sum():,}"
)

display(
    review_summary
    .sort_values("review_count", ascending=False)
    .head(10)
)

Original review rows     : 99,224
Aggregated review rows   : 98,673
Unique review order IDs  : 98,673
Duplicate order IDs      : 0


,order_id,review_count,unique_review_count,average_review_score,min_review_score,max_review_score,reviews_with_comments
1455,03c939fd7fd3b38f8485a0f95798f1f6,3,3,3.33,3,4,2
54489,8e17072ec97ce29f0e1f111e598b0c85,3,3,1.00,1,1,3
77319,c88b1d1b157a9999ce368f218a407141,3,3,4.33,3,5,0
86232,df56136b8031ecd28e200bb18e6ddb2e,3,3,5.00,5,5,0
48918,7f13a20e25350f4a55fb2a7c9a2e8d88,2,2,3.00,1,5,1
26449,44aa72f4eebc8c0ff7a3636f6369201b,2,2,3.00,3,3,1
47596,7b994fc7c7528d2098eebcb19d18ce7e,2,2,1.00,1,1,1
20315,34de5f7952fbd04a9a1e5d7001b014b5,2,2,1.00,1,1,2
38156,6318d8c5f6857de384c0b3186d036457,2,2,4.00,3,5,0
3765,09a38776c4e31230864f867821174daa,2,2,3.00,1,5,1


## 4.22 Integrate Order Review Summary

The aggregated review summary is joined to the integrated order-item dataset.

Because the review data has been reduced to one record per order, the join can
be performed as a many-to-one relationship without changing the order-item
grain.

A left join is used to preserve all existing order-item records, including
orders for which no customer review was submitted.

In [27]:
# Join aggregated review information

rows_before = len(integrated_df)

integrated_df = integrated_df.merge(
    review_summary,
    on="order_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

rows_after = len(integrated_df)

print(f"Rows before join : {rows_before:,}")
print(f"Rows after join  : {rows_after:,}")
print(f"Row difference   : {rows_after - rows_before:,}")

print("\nJoin result:")
print(integrated_df["_merge"].value_counts())

Rows before join : 112,650
Rows after join  : 112,650
Row difference   : 0

Join result:
_merge
both          111708
left_only        942
right_only         0
Name: count, dtype: int64


## 4.23 Analyse Orders Without Reviews

Order-item records without matching review information are analysed to determine
the number of affected orders.

The absence of a review is treated as expected customer behaviour rather than a
data quality exception, because customers are not required to submit a review
after completing an order.

In [28]:
# Analyse records without review information

unmatched_reviews = integrated_df[
    integrated_df["_merge"] == "left_only"
].copy()

print(f"Order-item rows without reviews : {len(unmatched_reviews):,}")
print(
    f"Unique orders without reviews   : "
    f"{unmatched_reviews['order_id'].nunique():,}"
)

print("\nOrder status:")
print(
    unmatched_reviews[
        ["order_id", "order_status"]
    ]
    .drop_duplicates()
    ["order_status"]
    .value_counts()
)

Order-item rows without reviews : 942
Unique orders without reviews   : 749

Order status:
order_status
delivered     646
shipped        75
canceled       17
processing      6
invoiced        5
Name: count, dtype: int64


## 4.24 Finalise Review Attributes

Missing review information is treated as expected business behaviour rather than
a data quality issue.

Count-based review attributes are set to zero where no review exists, while
review score attributes remain null because a missing review is not equivalent
to a score of zero.

A `has_review` indicator is added to distinguish reviewed and non-reviewed
orders explicitly.

In [29]:
# Create review indicator before removing merge column

integrated_df["has_review"] = (
    integrated_df["_merge"] == "both"
)

# Remove temporary merge indicator
integrated_df = integrated_df.drop(columns="_merge")

# Set review counts to zero where no review exists
review_count_columns = [
    "review_count",
    "unique_review_count",
    "reviews_with_comments"
]

integrated_df[review_count_columns] = (
    integrated_df[review_count_columns]
    .fillna(0)
    .astype(int)
)

print("Review field validation")
print("-" * 50)

print(
    f"Rows with reviews    : "
    f"{integrated_df['has_review'].sum():,}"
)

print(
    f"Rows without reviews : "
    f"{(~integrated_df['has_review']).sum():,}"
)

print(
    f"Null average scores  : "
    f"{integrated_df['average_review_score'].isna().sum():,}"
)

Review field validation
--------------------------------------------------
Rows with reviews    : 111,708
Rows without reviews : 942
Null average scores  : 942


## 4.25 Analyse Geolocation Cardinality

The geolocation dataset is assessed before integration because ZIP-code prefixes
may occur multiple times with different latitude and longitude coordinates.

Although exact duplicate records were removed during data cleaning, the ZIP-code
prefix itself may still not be unique. Directly joining a non-unique geolocation
dataset to customers or sellers could therefore multiply analytical rows.

The cardinality of ZIP-code prefixes is analysed before defining an appropriate
geographical aggregation strategy.

In [30]:
# Analyse geolocation records by ZIP-code prefix

geo_df = datasets["geolocation_dataset"]

geo_cardinality = (
    geo_df
    .groupby("geolocation_zip_code_prefix")
    .size()
    .reset_index(name="geolocation_record_count")
)

print(f"Geolocation rows                 : {len(geo_df):,}")
print(
    f"Unique ZIP-code prefixes         : "
    f"{geo_df['geolocation_zip_code_prefix'].nunique():,}"
)
print(
    f"Maximum records for one ZIP code : "
    f"{geo_cardinality['geolocation_record_count'].max():,}"
)
print(
    f"ZIP codes with multiple records  : "
    f"{(geo_cardinality['geolocation_record_count'] > 1).sum():,}"
)

display(
    geo_cardinality
    .sort_values("geolocation_record_count", ascending=False)
    .head(10)
)

Geolocation rows                 : 738,332
Unique ZIP-code prefixes         : 19,015
Maximum records for one ZIP code : 779
ZIP codes with multiple records  : 17,823


,geolocation_zip_code_prefix,geolocation_record_count
9536,38400,779
8728,35500,751
4501,11680,727
4528,11740,678
9015,36400,627
9542,38408,621
9719,39400,620
8579,35162,611
9239,37200,596
8867,35900,589


## 4.26 Validate Geographic Consistency by ZIP-Code Prefix

ZIP-code prefixes containing multiple geolocation records are examined to
determine whether they consistently map to the same city and state.

This validation is performed before geographical aggregation because coordinate
records can safely be summarised only after understanding whether each ZIP-code
prefix represents a consistent geographic area.

The number of distinct cities and states associated with each ZIP-code prefix
is therefore calculated.

In [31]:
# Analyse city/state consistency within each ZIP-code prefix

geo_consistency = (
    geo_df
    .groupby("geolocation_zip_code_prefix")
    .agg(
        record_count=("geolocation_zip_code_prefix", "size"),
        city_count=("geolocation_city", "nunique"),
        state_count=("geolocation_state", "nunique")
    )
    .reset_index()
)

print(f"ZIP-code prefixes analysed       : {len(geo_consistency):,}")

print(
    f"ZIP codes linked to >1 city      : "
    f"{(geo_consistency['city_count'] > 1).sum():,}"
)

print(
    f"ZIP codes linked to >1 state     : "
    f"{(geo_consistency['state_count'] > 1).sum():,}"
)

display(
    geo_consistency[
        (geo_consistency["city_count"] > 1) |
        (geo_consistency["state_count"] > 1)
    ]
    .sort_values(
        ["state_count", "city_count", "record_count"],
        ascending=False
    )
    .head(20)
)

ZIP-code prefixes analysed       : 19,015
ZIP codes linked to >1 city      : 8,555
ZIP codes linked to >1 state     : 8


,geolocation_zip_code_prefix,record_count,city_count,state_count
14664,72915,10,3,2
16256,80630,77,2,2
15879,78557,76,2,2
1668,4011,70,2,2
367,2116,11,2,2
16146,79750,152,1,2
6506,21550,145,1,2
6696,23056,31,1,2
3443,6900,388,5,1
7726,28950,309,5,1


## 4.27 Create Aggregated Geolocation Lookup

The geolocation dataset contains multiple coordinate observations for many
ZIP-code prefixes and therefore cannot be joined directly to customer or seller
records without creating a many-to-many relationship and potentially
multiplying rows.

A one-row-per-ZIP-code lookup is created before integration.

For each ZIP-code prefix:

- median latitude is used as the representative latitude;
- median longitude is used as the representative longitude;
- the most frequently occurring city is selected;
- the most frequently occurring state is selected.

This produces a unique ZIP-code-level geographic lookup suitable for
many-to-one joins while preserving representative geographic information.

In [32]:
# Create one-row-per-ZIP geolocation lookup

def most_frequent(series):
    """Return the most frequently occurring non-null value."""
    mode_values = series.dropna().mode()

    if len(mode_values) == 0:
        return pd.NA

    return mode_values.iloc[0]


geo_lookup = (
    geo_df
    .groupby("geolocation_zip_code_prefix")
    .agg(
        geolocation_lat=("geolocation_lat", "median"),
        geolocation_lng=("geolocation_lng", "median"),
        geolocation_city=("geolocation_city", most_frequent),
        geolocation_state=("geolocation_state", most_frequent)
    )
    .reset_index()
)

print(f"Original geolocation rows : {len(geo_df):,}")
print(f"Aggregated lookup rows    : {len(geo_lookup):,}")
print(
    f"Unique ZIP-code prefixes  : "
    f"{geo_lookup['geolocation_zip_code_prefix'].nunique():,}"
)

print(
    f"Duplicate ZIP keys        : "
    f"{geo_lookup['geolocation_zip_code_prefix'].duplicated().sum():,}"
)

display(geo_lookup.head(10))

Original geolocation rows : 738,332
Aggregated lookup rows    : 19,015
Unique ZIP-code prefixes  : 19,015
Duplicate ZIP keys        : 0


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1001,-23.549951,-46.634027,sao paulo,SP
1,1002,-23.548228,-46.635247,sao paulo,SP
2,1003,-23.548977,-46.635313,sao paulo,SP
3,1004,-23.549550,-46.634771,sao paulo,SP
4,1005,-23.549763,-46.636100,sao paulo,SP
5,1006,-23.550317,-46.636225,sao paulo,SP
6,1007,-23.550349,-46.637342,sao paulo,SP
7,1008,-23.545450,-46.635610,sao paulo,SP
8,1009,-23.546341,-46.636239,sao paulo,SP
9,1010,-23.546686,-46.635460,sao paulo,SP


## 4.28 Enrich Customer Data with Geolocation

The aggregated geolocation lookup is joined to the customer dataset using the
customer ZIP-code prefix.

Because the geolocation lookup contains exactly one record per ZIP-code prefix,
the relationship is many-to-one and does not introduce row multiplication.

Geographic columns are renamed with a `customer_` prefix to distinguish them
from seller geographic attributes that will be added separately.

In [33]:
# Enrich customer data with geographic information

customers_df = datasets["customers_dataset"].copy()

rows_before = len(customers_df)

customers_geo = customers_df.merge(
    geo_lookup,
    left_on="customer_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left",
    validate="many_to_one",
    indicator=True
)

rows_after = len(customers_geo)

print(f"Customer rows before join : {rows_before:,}")
print(f"Customer rows after join  : {rows_after:,}")
print(f"Row difference            : {rows_after - rows_before:,}")

print("\nJoin result:")
print(customers_geo["_merge"].value_counts())

Customer rows before join : 99,441
Customer rows after join  : 99,441
Row difference            : 0

Join result:
_merge
both          99163
left_only       278
right_only        0
Name: count, dtype: int64


## 4.29 Investigate Unmatched Customer Geolocation Records

The customer-to-geolocation join retained the original customer grain of
99,441 records, confirming that the aggregated geolocation lookup did not
introduce row multiplication.

A total of 99,163 customer records matched a ZIP-code prefix in the
geolocation lookup, while 278 customer records had no corresponding
geolocation record.

The unmatched records are retained because the absence of geographic
reference data should not result in the loss of valid customer or
transactional records. Their geographic attributes will remain null and
can be treated as a data quality limitation in downstream analysis.

In [34]:
# Investigate customers without a geolocation match

unmatched_customers_geo = customers_geo[
    customers_geo["_merge"] == "left_only"
].copy()

print(f"Unmatched customer rows       : {len(unmatched_customers_geo):,}")
print(
    f"Unique unmatched ZIP prefixes : "
    f"{unmatched_customers_geo['customer_zip_code_prefix'].nunique():,}"
)

display(
    unmatched_customers_geo[
        [
            "customer_id",
            "customer_unique_id",
            "customer_zip_code_prefix",
            "customer_city",
            "customer_state"
        ]
    ].head(20)
)

Unmatched customer rows       : 278
Unique unmatched ZIP prefixes : 157


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
354,ecb1725b26e8b8c458181455dfa434ea,b55a113bb84fc10eaf58c6d09ec69794,72300,brasilia,DF
382,bcf86029aeed4ed8bac0e16eb14c22f5,7cd7974c9f79f75b77f323878ef87f43,11547,cubatao,SP
877,f4302056f0c58570522590f8181de2c7,67b05b597a66b5c449025000b9430abb,64605,picos,PI
1218,03bbe0ce5c28e05f22917607db798818,8f3dca4306d5a89e4ae2c65c110603a2,72465,brasilia,DF
1272,ad4950aded55c2ea376be59506456d68,aa2b96dd03307ea6dc4b763c0b5f0b39,7729,caieiras,SP
1309,6e7dc8f6ec3f0a0d72fa64992ebe2fc8,5efeef1510d928c549c082d0ef4447ac,72904,santo antonio do descoberto,GO
1421,c55a17a7c31353c35d48550b3aebc06f,578d5e6d37a20bbf631183feedfe2dea,35408,glaura,MG
1742,3a9686af66e7ba1291b19d2e41d584ce,c85df1c6ef6f7bb605e1c49d34f172ee,78554,sinop,MT
2095,baca33004aa726524d5a891853100559,aa67457bb07fde4e952d67b038135bee,73369,brasilia,DF
2166,7557541c9c578082c892cf185c3b0b47,7ee645e84a436c9dad3d5af253162fe7,8980,nossa senhora do remedio,SP


## 4.30 Prepare Customer Geography for Integration

Customer geographic attributes are prepared for integration into the
order-item analytical dataset.

The geolocation lookup successfully matched the majority of customer records.
Customers without a corresponding geolocation ZIP-code record are retained,
with latitude and longitude left as null rather than inferred or fabricated.

Geolocation-derived fields are renamed with a `customer_geo_` prefix to
distinguish them from the original customer location attributes and from
seller geography that will be introduced later.

In [35]:
# Prepare enriched customer dataset for integration

customers_geo = customers_geo.drop(columns=["_merge"])

customers_geo = customers_geo.rename(
    columns={
        "geolocation_lat": "customer_geo_lat",
        "geolocation_lng": "customer_geo_lng",
        "geolocation_city": "customer_geo_city",
        "geolocation_state": "customer_geo_state"
    }
)

# The lookup ZIP key is redundant after the join
customers_geo = customers_geo.drop(
    columns=["geolocation_zip_code_prefix"]
)

print(f"Customer rows              : {len(customers_geo):,}")
print(f"Unique customer IDs        : {customers_geo['customer_id'].nunique():,}")
print(
    f"Missing customer latitude  : "
    f"{customers_geo['customer_geo_lat'].isna().sum():,}"
)
print(
    f"Missing customer longitude : "
    f"{customers_geo['customer_geo_lng'].isna().sum():,}"
)

display(customers_geo.head())

Customer rows              : 99,441
Unique customer IDs        : 99,441
Missing customer latitude  : 278
Missing customer longitude : 278


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,customer_geo_lat,customer_geo_lng,customer_geo_city,customer_geo_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,-20.502307,-47.396740,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,-23.730435,-46.541474,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,-23.531294,-46.656980,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,-23.499025,-46.183436,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,-22.974331,-47.142173,campinas,SP


## 4.31 Enrich Seller Data with Geolocation

The aggregated geolocation lookup is joined to the seller dataset using the
seller ZIP-code prefix.

The one-row-per-ZIP geolocation lookup ensures that this is a many-to-one
relationship and prevents row multiplication.

The join is validated before seller geographic attributes are incorporated
into the integrated analytical dataset.

In [36]:
# Enrich seller data with geographic information

sellers_df = datasets["sellers_dataset"].copy()

rows_before = len(sellers_df)

sellers_geo = sellers_df.merge(
    geo_lookup,
    left_on="seller_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left",
    validate="many_to_one",
    indicator=True
)

rows_after = len(sellers_geo)

print(f"Seller rows before join : {rows_before:,}")
print(f"Seller rows after join  : {rows_after:,}")
print(f"Row difference          : {rows_after - rows_before:,}")

print("\nJoin result:")
print(sellers_geo["_merge"].value_counts())

Seller rows before join : 3,095
Seller rows after join  : 3,095
Row difference          : 0

Join result:
_merge
both          3088
left_only        7
right_only       0
Name: count, dtype: int64


## 4.32 Prepare Seller Geography for Integration

The seller-to-geolocation join retained all 3,095 seller records without
introducing row multiplication.

A total of 3,088 sellers matched the geolocation lookup, while 7 sellers had
no corresponding ZIP-code reference record. These sellers are retained because
the absence of reference geography should not result in the removal of valid
seller records.

Geolocation-derived attributes are renamed with a `seller_geo_` prefix to
distinguish them from the original seller location fields and customer
geographic attributes.

In [37]:
# Prepare enriched seller dataset for integration

sellers_geo = sellers_geo.drop(columns=["_merge"])

sellers_geo = sellers_geo.rename(
    columns={
        "geolocation_lat": "seller_geo_lat",
        "geolocation_lng": "seller_geo_lng",
        "geolocation_city": "seller_geo_city",
        "geolocation_state": "seller_geo_state"
    }
)

# Remove redundant lookup ZIP key
sellers_geo = sellers_geo.drop(
    columns=["geolocation_zip_code_prefix"]
)

print(f"Seller rows              : {len(sellers_geo):,}")
print(f"Unique seller IDs        : {sellers_geo['seller_id'].nunique():,}")
print(
    f"Missing seller latitude  : "
    f"{sellers_geo['seller_geo_lat'].isna().sum():,}"
)
print(
    f"Missing seller longitude : "
    f"{sellers_geo['seller_geo_lng'].isna().sum():,}"
)

display(sellers_geo.head())

Seller rows              : 3,095
Unique seller IDs        : 3,095
Missing seller latitude  : 7
Missing seller longitude : 7


,seller_id,seller_zip_code_prefix,seller_city,seller_state,seller_geo_lat,seller_geo_lng,seller_geo_city,seller_geo_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP,-22.893863,-47.062006,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP,-22.382869,-46.947992,mogi-guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ,-22.910174,-43.176775,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP,-23.656991,-46.612490,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP,-22.963763,-46.534676,braganca paulista,SP


## 4.33 Join Customer Geography to Integrated Dataset

Customer geographic attributes are added to the order-item analytical dataset.

Only the newly derived geographic fields are selected from the enriched
customer dataset because the core customer attributes were already introduced
during the earlier customer join.

The join uses `customer_id` as the key and is validated as many-to-one to
preserve the order-item grain of the integrated dataset.

In [38]:
# Select customer geographic attributes required for integration

customer_geo_fields = customers_geo[
    [
        "customer_id",
        "customer_geo_lat",
        "customer_geo_lng",
        "customer_geo_city",
        "customer_geo_state"
    ]
].copy()

# Validate customer key before joining
print(
    f"Duplicate customer IDs in geography lookup : "
    f"{customer_geo_fields['customer_id'].duplicated().sum():,}"
)

rows_before = len(integrated_df)

integrated_df = integrated_df.merge(
    customer_geo_fields,
    on="customer_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

rows_after = len(integrated_df)

print(f"\nRows before join : {rows_before:,}")
print(f"Rows after join  : {rows_after:,}")
print(f"Row difference   : {rows_after - rows_before:,}")

print("\nJoin result:")
print(integrated_df["_merge"].value_counts())

Duplicate customer IDs in geography lookup : 0

Rows before join : 112,650
Rows after join  : 112,650
Row difference   : 0

Join result:
_merge
both          112650
left_only          0
right_only         0
Name: count, dtype: int64


## 4.34 Validate Customer Geography Coverage

Customer geography coverage is assessed after integration.

Although every order-item record successfully matched a customer record,
some customers did not have a corresponding ZIP-code entry in the
geolocation reference dataset. These records are retained with null
geographic coordinates.

The validation quantifies the impact of the missing geolocation data at
the order-item grain.

In [39]:
# Remove merge indicator from previous join
integrated_df = integrated_df.drop(columns="_merge")

# Validate customer geography coverage at order-item grain

missing_customer_geo = integrated_df[
    integrated_df["customer_geo_lat"].isna()
].copy()

print(
    f"Order-item rows missing customer geography : "
    f"{len(missing_customer_geo):,}"
)

print(
    f"Unique orders affected                    : "
    f"{missing_customer_geo['order_id'].nunique():,}"
)

print(
    f"Unique customers affected                 : "
    f"{missing_customer_geo['customer_id'].nunique():,}"
)

print(
    f"Percentage of order-item rows affected     : "
    f"{len(missing_customer_geo) / len(integrated_df) * 100:.2f}%"
)

Order-item rows missing customer geography : 302
Unique orders affected                    : 274
Unique customers affected                 : 274
Percentage of order-item rows affected     : 0.27%


## 4.35 Join Seller Geography to Integrated Dataset

Seller geographic attributes are added to the order-item analytical dataset.

Only the derived geographic fields are selected because the core seller
attributes were already incorporated during the earlier seller join.

The relationship is validated as many-to-one using `seller_id`, ensuring
that the existing order-item grain is preserved.

In [40]:
# Select seller geographic attributes required for integration

seller_geo_fields = sellers_geo[
    [
        "seller_id",
        "seller_geo_lat",
        "seller_geo_lng",
        "seller_geo_city",
        "seller_geo_state"
    ]
].copy()

# Validate seller key before joining
print(
    f"Duplicate seller IDs in geography lookup : "
    f"{seller_geo_fields['seller_id'].duplicated().sum():,}"
)

rows_before = len(integrated_df)

integrated_df = integrated_df.merge(
    seller_geo_fields,
    on="seller_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

rows_after = len(integrated_df)

print(f"\nRows before join : {rows_before:,}")
print(f"Rows after join  : {rows_after:,}")
print(f"Row difference   : {rows_after - rows_before:,}")

print("\nJoin result:")
print(integrated_df["_merge"].value_counts())

Duplicate seller IDs in geography lookup : 0

Rows before join : 112,650
Rows after join  : 112,650
Row difference   : 0

Join result:
_merge
both          112650
left_only          0
right_only         0
Name: count, dtype: int64


## 4.36 Validate Seller Geography Coverage

Seller geography coverage is assessed after integration.

All order-item records successfully matched a seller record. However, a small
number of sellers did not have a corresponding ZIP-code entry in the
geolocation reference dataset.

These records are retained with null geographic coordinates rather than being
removed or artificially populated. The validation below quantifies the impact
of missing seller geography at the order-item grain.

In [41]:
# Remove merge indicator from previous join
integrated_df = integrated_df.drop(columns="_merge")

# Validate seller geography coverage at order-item grain

missing_seller_geo = integrated_df[
    integrated_df["seller_geo_lat"].isna()
].copy()

print(
    f"Order-item rows missing seller geography : "
    f"{len(missing_seller_geo):,}"
)

print(
    f"Unique orders affected                  : "
    f"{missing_seller_geo['order_id'].nunique():,}"
)

print(
    f"Unique sellers affected                 : "
    f"{missing_seller_geo['seller_id'].nunique():,}"
)

print(
    f"Percentage of order-item rows affected   : "
    f"{len(missing_seller_geo) / len(integrated_df) * 100:.2f}%"
)

Order-item rows missing seller geography : 253
Unique orders affected                  : 220
Unique sellers affected                 : 7
Percentage of order-item rows affected   : 0.22%


## 4.37 Create Core Analytical Features

Derived analytical features are created from the integrated source attributes
to support downstream reporting and analysis.

The features include:

- total order-item value including freight;
- delivery duration in days;
- delivery performance against the estimated delivery date;
- customer-to-seller geographic distance where valid coordinates are available.

Derived fields are calculated without changing the established order-item
grain of the dataset.

In [42]:
# Create core analytical features

import numpy as np

# Total value of each order item including allocated freight
integrated_df["item_total_value"] = (
    integrated_df["price"] + integrated_df["freight_value"]
)

# Delivery duration in days
integrated_df["delivery_days"] = (
    integrated_df["order_delivered_customer_date"]
    - integrated_df["order_purchase_timestamp"]
).dt.total_seconds() / 86400

# Difference between actual and estimated delivery
integrated_df["delivery_vs_estimate_days"] = (
    integrated_df["order_delivered_customer_date"]
    - integrated_df["order_estimated_delivery_date"]
).dt.total_seconds() / 86400

# Delivery status
integrated_df["delivery_performance"] = np.select(
    [
        integrated_df["order_delivered_customer_date"].isna(),
        integrated_df["delivery_vs_estimate_days"] <= 0,
        integrated_df["delivery_vs_estimate_days"] > 0
    ],
    [
        "Not delivered / date unavailable",
        "On or before estimate",
        "Late"
    ],
    default="Unknown"
)

print(f"Rows after feature creation : {len(integrated_df):,}")

display(
    integrated_df[
        [
            "order_id",
            "order_item_id",
            "price",
            "freight_value",
            "item_total_value",
            "delivery_days",
            "delivery_vs_estimate_days",
            "delivery_performance"
        ]
    ].head(10)
)

TypeError: operation 'sub' not supported for dtype 'str' with dtype 'str'

In [43]:
# Create core analytical features

import numpy as np

# Ensure required order date fields are datetime
date_columns = [
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    integrated_df[col] = pd.to_datetime(
        integrated_df[col],
        errors="coerce"
    )

# Confirm data types
print("Date column data types:")
print(integrated_df[date_columns].dtypes)

# Total value of each order item including freight
integrated_df["item_total_value"] = (
    integrated_df["price"] + integrated_df["freight_value"]
)

# Delivery duration in days
integrated_df["delivery_days"] = (
    integrated_df["order_delivered_customer_date"]
    - integrated_df["order_purchase_timestamp"]
).dt.total_seconds() / 86400

# Difference between actual and estimated delivery
integrated_df["delivery_vs_estimate_days"] = (
    integrated_df["order_delivered_customer_date"]
    - integrated_df["order_estimated_delivery_date"]
).dt.total_seconds() / 86400

# Classify delivery performance
integrated_df["delivery_performance"] = np.select(
    [
        integrated_df["order_delivered_customer_date"].isna(),
        integrated_df["delivery_vs_estimate_days"] <= 0,
        integrated_df["delivery_vs_estimate_days"] > 0
    ],
    [
        "Not delivered / date unavailable",
        "On or before estimate",
        "Late"
    ],
    default="Unknown"
)

print(f"\nRows after feature creation : {len(integrated_df):,}")

display(
    integrated_df[
        [
            "order_id",
            "order_item_id",
            "price",
            "freight_value",
            "item_total_value",
            "delivery_days",
            "delivery_vs_estimate_days",
            "delivery_performance"
        ]
    ].head(10)
)

Date column data types:
order_purchase_timestamp         datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

Rows after feature creation : 112,650


,order_id,order_item_id,price,freight_value,item_total_value,delivery_days,delivery_vs_estimate_days,delivery_performance
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29,72.19,7.614421,-8.011250,On or before estimate
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93,259.83,16.216181,-2.330278,On or before estimate
2,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87,216.87,7.948437,-13.444954,On or before estimate
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79,25.78,6.147269,-5.435660,On or before estimate
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14,218.04,25.114352,-15.303808,On or before estimate
5,00048cc3ae777c65dbb7d2a0634bc1ea,1,21.90,12.69,34.59,6.668067,-14.427373,On or before estimate
6,00054e8431b9d7675808bcb819fb4a32,1,19.90,11.85,31.75,8.423495,-16.080810,On or before estimate
7,000576fe39319847cbb9d288c5617fa6,1,810.00,70.75,880.75,5.080324,-15.413808,On or before estimate
8,0005a1a1728c9d785b8e2b08b904576c,1,145.95,11.65,157.60,9.984005,0.762164,Late
9,0005f50442cb953dcd1d21e1fb923495,1,53.99,11.40,65.39,2.145046,-18.271863,On or before estimate


## 4.38 Validate Geographic Coordinates

Customer and seller coordinates are validated before geographic distance
calculations are performed.

Valid geographic coordinates must fall within the following ranges:

- latitude: -90 to 90 degrees;
- longitude: -180 to 180 degrees.

Null coordinates identified during the earlier geolocation matching process
are retained and are not treated as invalid values.

This validation prevents invalid coordinate values from being used in
customer-to-seller distance calculations.

In [44]:
# Validate customer and seller geographic coordinates

coordinate_validation = pd.DataFrame({
    "coordinate": [
        "customer_geo_lat",
        "customer_geo_lng",
        "seller_geo_lat",
        "seller_geo_lng"
    ],
    "valid_min": [
        -90,
        -180,
        -90,
        -180
    ],
    "valid_max": [
        90,
        180,
        90,
        180
    ]
})

validation_results = []

for _, row in coordinate_validation.iterrows():

    col = row["coordinate"]
    valid_min = row["valid_min"]
    valid_max = row["valid_max"]

    values = pd.to_numeric(
        integrated_df[col],
        errors="coerce"
    )

    invalid_count = (
        values.notna()
        & ~values.between(valid_min, valid_max)
    ).sum()

    validation_results.append({
        "coordinate": col,
        "minimum_value": values.min(),
        "maximum_value": values.max(),
        "null_values": values.isna().sum(),
        "invalid_values": invalid_count
    })

coordinate_validation_df = pd.DataFrame(validation_results)

display(coordinate_validation_df)

,coordinate,minimum_value,maximum_value,null_values,invalid_values
0,customer_geo_lat,-33.690729,42.184003,302,0
1,customer_geo_lng,-72.670621,-8.723762,302,0
2,seller_geo_lat,-32.073840,-2.501703,253,0
3,seller_geo_lng,-67.812015,-34.855504,253,0


## 4.39 Calculate Customer-to-Seller Distance

The geographic distance between each customer and seller is calculated using
their latitude and longitude coordinates.

The Haversine formula is used to estimate the great-circle distance between
two points on the Earth's surface.

Distance is expressed in kilometres and provides a useful analytical measure
for logistics, freight, delivery-time and regional analysis.

Where either customer or seller coordinates are unavailable, the calculated
distance remains null rather than being estimated.

In [45]:
# Calculate customer-to-seller distance using the Haversine formula

# Convert coordinate columns to numeric values
customer_lat = pd.to_numeric(
    integrated_df["customer_geo_lat"],
    errors="coerce"
)

customer_lng = pd.to_numeric(
    integrated_df["customer_geo_lng"],
    errors="coerce"
)

seller_lat = pd.to_numeric(
    integrated_df["seller_geo_lat"],
    errors="coerce"
)

seller_lng = pd.to_numeric(
    integrated_df["seller_geo_lng"],
    errors="coerce"
)

# Convert degrees to radians
lat1 = np.radians(customer_lat)
lon1 = np.radians(customer_lng)
lat2 = np.radians(seller_lat)
lon2 = np.radians(seller_lng)

# Differences between coordinates
delta_lat = lat2 - lat1
delta_lon = lon2 - lon1

# Haversine formula
a = (
    np.sin(delta_lat / 2) ** 2
    + np.cos(lat1)
    * np.cos(lat2)
    * np.sin(delta_lon / 2) ** 2
)

c = 2 * np.arcsin(np.sqrt(a))

# Mean radius of Earth in kilometres
earth_radius_km = 6371.0088

integrated_df["customer_seller_distance_km"] = (
    earth_radius_km * c
)

print(
    f"Rows in integrated dataset       : "
    f"{len(integrated_df):,}"
)

print(
    f"Distances calculated             : "
    f"{integrated_df['customer_seller_distance_km'].notna().sum():,}"
)

print(
    f"Distances unavailable            : "
    f"{integrated_df['customer_seller_distance_km'].isna().sum():,}"
)

print(
    f"Average customer-seller distance : "
    f"{integrated_df['customer_seller_distance_km'].mean():,.2f} km"
)

print(
    f"Median customer-seller distance  : "
    f"{integrated_df['customer_seller_distance_km'].median():,.2f} km"
)

print(
    f"Maximum customer-seller distance : "
    f"{integrated_df['customer_seller_distance_km'].max():,.2f} km"
)

Rows in integrated dataset       : 112,650
Distances calculated             : 112,096
Distances unavailable            : 554
Average customer-seller distance : 596.90 km
Median customer-seller distance  : 431.72 km
Maximum customer-seller distance : 8,677.91 km


## 4.40 Validate Customer-to-Seller Distance

The calculated customer-to-seller distance is validated to identify unusual
or potentially erroneous geographic results.

Summary statistics and the largest calculated distances are reviewed to
confirm that the derived feature is plausible before it is used in subsequent
analysis.

Large distances are retained at this stage unless there is sufficient evidence
that they result from invalid source data.

In [46]:
# Validate calculated customer-to-seller distances

distance_validation = pd.DataFrame({
    "metric": [
        "Total rows",
        "Distances calculated",
        "Distances unavailable",
        "Minimum distance (km)",
        "Median distance (km)",
        "Average distance (km)",
        "Maximum distance (km)"
    ],
    "value": [
        len(integrated_df),
        integrated_df["customer_seller_distance_km"].notna().sum(),
        integrated_df["customer_seller_distance_km"].isna().sum(),
        integrated_df["customer_seller_distance_km"].min(),
        integrated_df["customer_seller_distance_km"].median(),
        integrated_df["customer_seller_distance_km"].mean(),
        integrated_df["customer_seller_distance_km"].max()
    ]
})

display(distance_validation)

,metric,value
0,Total rows,112650.000000
1,Distances calculated,112096.000000
2,Distances unavailable,554.000000
3,Minimum distance (km),0.000000
4,Median distance (km),431.717812
5,Average distance (km),596.904641
6,Maximum distance (km),8677.910925


In [47]:
# Inspect the 20 largest customer-to-seller distances

largest_distances = (
    integrated_df[
        [
            "order_id",
            "customer_city",
            "customer_state",
            "seller_city",
            "seller_state",
            "customer_geo_lat",
            "customer_geo_lng",
            "seller_geo_lat",
            "seller_geo_lng",
            "customer_seller_distance_km"
        ]
    ]
    .sort_values(
        "customer_seller_distance_km",
        ascending=False
    )
    .head(20)
)

display(largest_distances)

,order_id,customer_city,customer_state,seller_city,seller_state,customer_geo_lat,customer_geo_lng,seller_geo_lat,seller_geo_lng,customer_seller_distance_km
60915,8ad3f1d0f96992e43566c4c82c9f6c58,ilha dos valadares,PR,sao bento do sul,SC,42.184003,-8.723762,-26.231963,-49.412079,8677.910925
75917,acdbc7396e191931c263db11af241d62,porto trombetas,PA,ribeirao preto,SP,19.836719,-30.411009,-21.259301,-47.815144,4946.140616
34191,4d5abe7999d76d1fb6237d3677706af0,porto trombetas,PA,ribeirao preto,SP,19.836719,-30.411009,-21.259301,-47.815144,4946.140616
103437,eaec582d6403b30186964f6df957047e,pau d'arco,AL,ibitinga,SP,10.903855,-18.982951,-21.757538,-48.829262,4878.155992
60328,897ec6416d50126a9061626f0fc2d658,ibiajara,BA,sao paulo,SP,13.001420,-23.583939,-23.652774,-46.755657,4791.652898
34929,4f1583d080fe1eec5e509335d79b17c6,santo antonio do canaa,ES,mogi das cruzes,SP,0.916934,-69.569797,-23.523237,-46.194437,3710.425298
102832,e9874f4e48ede77b6b9d785ac3755445,boa vista,RR,bombinhas,SC,2.821374,-60.672180,-27.152078,-48.505384,3579.266994
68005,9b790ca446b8e28ce5e4382959f6b7df,boa vista,RR,fazenda rio grande,PR,2.817039,-60.755409,-25.671852,-49.327451,3398.552914
17682,2889d7816934298a0f5a4a30ef3ffbc7,boa vista,RR,sao jose dos pinhais,PR,2.808802,-60.679217,-25.543616,-49.182564,3387.264170
104196,ecab90c9933c58908d3d6add7c6f5ae3,boa vista,RR,curitiba,PR,2.817039,-60.755409,-25.500399,-49.335452,3380.684648


### Distance Validation Finding

Review of the largest calculated customer-to-seller distances identified
geographic anomalies in the source geolocation data.

For example, some Brazilian customer locations are associated with coordinates
that are geographically inconsistent with their recorded city and state.
Consequently, extreme distance values may reflect source-data geolocation
quality issues rather than genuine transport distances.

The Haversine calculation itself is retained because the calculation is
technically valid. No records are removed or coordinates manually corrected,
as there is insufficient evidence to infer replacement coordinates reliably.

These extreme values are therefore retained as known source-data limitations
and should be interpreted cautiously in downstream analysis.

## 4.41 Final Integrated Dataset Validation

The completed integrated dataset is validated before export.

The checks confirm that the order-item grain has been preserved, no duplicate
order-item keys have been introduced, and the expected number of rows remains
unchanged following all enrichment and feature-engineering operations.

In [48]:
# Final validation of integrated analytical dataset

final_integration_validation = pd.DataFrame({
    "validation_check": [
        "Total integrated rows",
        "Unique order-item combinations",
        "Duplicate order-item combinations",
        "Expected order-item rows",
        "Row count preserved"
    ],
    "result": [
        len(integrated_df),

        integrated_df[
            ["order_id", "order_item_id"]
        ].drop_duplicates().shape[0],

        integrated_df.duplicated(
            subset=["order_id", "order_item_id"]
        ).sum(),

        112650,

        len(integrated_df) == 112650
    ]
})

display(final_integration_validation)

,validation_check,result
0,Total integrated rows,112650
1,Unique order-item combinations,112650
2,Duplicate order-item combinations,0
3,Expected order-item rows,112650
4,Row count preserved,True


## 4.42 Prepare Final Analytical Dataset

A final analytical dataset is created from the integrated data.

Temporary merge indicators and intermediate processing fields are excluded
where appropriate, while business attributes, data-quality flags, aggregated
measures and engineered analytical features are retained.

A separate DataFrame is created so that the completed integration dataset
remains available for audit and validation.

In [49]:
# Create final analytical dataset

analytical_df = integrated_df.copy()

# Remove temporary merge indicator if still present
if "_merge" in analytical_df.columns:
    analytical_df = analytical_df.drop(columns="_merge")

print(f"Final analytical rows    : {len(analytical_df):,}")
print(f"Final analytical columns : {len(analytical_df.columns):,}")
print(
    f"Duplicate order-item keys: "
    f"{analytical_df.duplicated(subset=['order_id', 'order_item_id']).sum():,}"
)

Final analytical rows    : 112,650
Final analytical columns : 60
Duplicate order-item keys: 0


In [50]:
# Display final analytical dataset structure

column_inventory = pd.DataFrame({
    "column_number": range(1, len(analytical_df.columns) + 1),
    "column_name": analytical_df.columns,
    "data_type": analytical_df.dtypes.astype(str).values
})

display(column_inventory)

display(analytical_df.head())

,column_number,column_name,data_type
0,1,order_id,str
1,2,order_item_id,int64
2,3,product_id,str
3,4,seller_id,str
4,5,shipping_limit_date,str
5,6,price,float64
6,7,freight_value,float64
7,8,customer_id,str
8,9,order_status,str
9,10,order_purchase_timestamp,datetime64[us]


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,order_purchase_timestamp,...,customer_geo_state,seller_geo_lat,seller_geo_lng,seller_geo_city,seller_geo_state,item_total_value,delivery_days,delivery_vs_estimate_days,delivery_performance,customer_seller_distance_km
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,...,RJ,-22.499217,-44.125997,volta redonda,RJ,72.19,7.614421,-8.011250,On or before estimate,301.299954
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,...,SP,-23.565803,-46.519192,sao paulo,SP,259.83,16.216181,-2.330278,On or before estimate,588.545069
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,...,MG,-22.272310,-46.165743,borda da mata,MG,216.87,7.948437,-13.444954,On or before estimate,312.955650
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,...,SP,-20.554801,-47.387749,franca,SP,25.78,6.147269,-5.435660,On or before estimate,295.319989
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,...,SP,-22.930512,-53.136361,loanda,PR,218.04,25.114352,-15.303808,On or before estimate,646.460714


In [51]:
# Standardise date columns in final analytical dataset

date_columns = [
    "shipping_limit_date",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    if col in analytical_df.columns:
        analytical_df[col] = pd.to_datetime(
            analytical_df[col],
            errors="coerce"
        )

# Confirm date data types
print("Final date column data types:\n")
print(analytical_df[date_columns].dtypes)

Final date column data types:

shipping_limit_date              datetime64[us]
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


## 4.43 Export Final Integrated Dataset

The validated analytical dataset is exported as the final output of the
integration process.

The dataset retains the order-item grain and combines order, customer,
product, seller, payment, review and geographic information together with
engineered analytical features and data-quality indicators.

The exported dataset provides a reusable analytical layer for subsequent
reporting and analysis.

In [52]:
# Export final integrated analytical dataset

final_output_path = output_data_path / "olist_integrated_analytical_dataset.csv"

analytical_df.to_csv(
    final_output_path,
    index=False
)

print(f"Export completed successfully.")
print(f"Output file : {final_output_path}")
print(f"Rows        : {len(analytical_df):,}")
print(f"Columns     : {len(analytical_df.columns):,}")

Export completed successfully.
Output file : ..\data\output\olist_integrated_analytical_dataset.csv
Rows        : 112,650
Columns     : 60


## 4.44 Verify Final Export

The exported analytical dataset is reloaded and validated to confirm that the
file was written successfully and that its dimensions and order-item grain
match the in-memory dataset.

In [53]:
# Reload and verify exported analytical dataset

export_check_df = pd.read_csv(final_output_path)

export_validation = pd.DataFrame({
    "validation_check": [
        "Export file exists",
        "Exported row count",
        "Expected row count",
        "Row count matches",
        "Exported column count",
        "Expected column count",
        "Column count matches",
        "Duplicate order-item keys"
    ],
    "result": [
        final_output_path.exists(),
        len(export_check_df),
        len(analytical_df),
        len(export_check_df) == len(analytical_df),
        len(export_check_df.columns),
        len(analytical_df.columns),
        len(export_check_df.columns) == len(analytical_df.columns),
        export_check_df.duplicated(
            subset=["order_id", "order_item_id"]
        ).sum()
    ]
})

display(export_validation)

,validation_check,result
0,Export file exists,True
1,Exported row count,112650
2,Expected row count,112650
3,Row count matches,True
4,Exported column count,60
5,Expected column count,60
6,Column count matches,True
7,Duplicate order-item keys,0


In [54]:
# Export canonical analytical dataset in Parquet format

parquet_output_path = (
    output_data_path /
    "olist_integrated_analytical_dataset.parquet"
)

analytical_df.to_parquet(
    parquet_output_path,
    index=False,
    engine="pyarrow"
)

print("Parquet export completed successfully.")
print(f"Output file : {parquet_output_path}")
print(f"Rows        : {len(analytical_df):,}")
print(f"Columns     : {len(analytical_df.columns):,}")

Parquet export completed successfully.
Output file : ..\data\output\olist_integrated_analytical_dataset.parquet
Rows        : 112,650
Columns     : 60


In [55]:
# Verify Parquet output

parquet_check_df = pd.read_parquet(
    parquet_output_path,
    engine="pyarrow"
)

print(f"Rows       : {len(parquet_check_df):,}")
print(f"Columns    : {len(parquet_check_df.columns):,}")

print(
    "Duplicate order-item keys:",
    parquet_check_df.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum()
)

print(
    "Schema preserved:",
    analytical_df.dtypes.equals(parquet_check_df.dtypes)
)

Rows       : 112,650
Columns    : 60
Duplicate order-item keys: 0
Schema preserved: True


## 4.45 Integration Summary

The data integration phase combined the cleaned Olist source datasets into a
single analytical dataset at **order-item grain**.

### Integration Approach

The `order_items` dataset was selected as the base table because an order may
contain multiple products and sellers. Maintaining one row per
`order_id` + `order_item_id` preserves this level of analytical detail.

Before integration, relationship cardinality was assessed to prevent
many-to-many joins and unintended row multiplication.

Customer, order, product and seller attributes were integrated using validated
many-to-one relationships.

### Payment and Review Integration

Payments and reviews required additional treatment because individual orders
may contain multiple payment or review records.

These datasets were aggregated to one record per order before joining to the
order-item dataset. This preserved the required analytical grain while retaining
useful measures such as total payment value, payment characteristics and review
statistics.

### Geographic Enrichment

The geolocation dataset contained multiple records for many ZIP-code prefixes.
A one-record-per-ZIP lookup was therefore created before enriching customer and
seller records.

Some customer and seller ZIP codes had no matching geolocation record. These
were retained as missing geographic values rather than being artificially
imputed.

Customer-to-seller geographic distance was derived using the Haversine formula.
Validation identified several extreme distances associated with geographically
inconsistent source coordinates. These records were retained and documented as
known source-data limitations rather than manually corrected without sufficient
evidence.

### Data Quality and Feature Engineering

Known source-data exceptions were retained through explicit data-quality flags,
including missing delivery dates, missing payments and invalid product-weight
conditions.

Additional analytical features were created to support downstream analysis,
including order-item value, delivery duration, delivery performance and
customer-to-seller distance.

### Final Validation

The completed integration preserved the intended order-item grain:

- Final rows: **112,650**
- Final columns: **60**
- Unique order-item combinations: **112,650**
- Duplicate order-item combinations: **0**
- Export validation: **Passed**

The final dataset was exported as:

`olist_integrated_analytical_dataset.csv`

The resulting dataset provides a validated analytical layer suitable for
downstream exploratory analysis, reporting and visualisation.